In [70]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage,BaseMessage
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv
from typing import TypedDict,Annotated

In [71]:
load_dotenv()
model = ChatOpenAI()
class Query(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]


In [72]:
def ExecutionQuery(state:Query):
    messages = state["messages"]
    response = model.invoke(messages)

    return {"messages":[response]}


In [73]:
checkpoint = MemorySaver()
state = StateGraph(Query)
state.add_node("ExecutionQuery",ExecutionQuery)
state.add_edge(START,"ExecutionQuery")
state.add_edge("ExecutionQuery",END)

workflow = state.compile(checkpointer=checkpoint)

In [74]:
# initial_state = {
#     "messages" : [HumanMessage(content="Who is Father of Jeff Besos")]
# }

# workflow.invoke(initial_state)


In [75]:
thread_id ="1"
while True:
    user_query = input("Type here your Question ")
    print("User Message",user_query)
    if user_query.strip().lower() in ["exit","bye","stop","quit"]:
        break

    config = {"configurable":{"thread_id":thread_id}}

    message = workflow.invoke({
        "messages" : [HumanMessage(content=user_query)]

    }, config=config)

    print("AI ",message["messages"][-1].content)

User Message whats my name
AI  I'm sorry, but I do not have access to personal information about users. Can I help you with anything else?
User Message 2 multiply by 5
AI  2 multiplied by 5 is equal to 10.
User Message 2 mutiply result
AI  I'm not sure what you mean by "2 mutiply result." Can you please provide more clarification or context so I can assist you better?
User Message result of previous answer
AI  The result of 2 multiplied by 5 is 10.
User Message now multiply result with 4
AI  10 (result of 2 multiplied by 5) multiplied by 4 is equal to 40.
User Message quit
